# 解析量子核与有限测量：最小可运行示例

这是 C08—C09 的一个数学示例。所有输入和标签均为**明确的解析构造**，没有股票、收益率、实验室资料或账户数据。这里的态是理想纯态，RY 与 CNOT 使其振幅为实数；结果来自本地 NumPy 计算，不是量子设备读数。

本示例检查三件事：

1. 4比特态的保真度核，与136维实对称密度特征的内积一致。
2. 独立有限采样可能让原本半正定的训练核变成不定矩阵。
3. 训练谱修正必须与评价交叉核一致，然后才能按固定的平均损失约定拟合。

各代码单元均自包含，可分别运行。只需已有的 NumPy；没有安装、联网或文件写入操作。保存的输出由独立新进程逐单元执行取得，**未据此验证完整 Jupyter 环境**。从课程根目录或本目录运行。协议依据见[数据与实验协议](../docs/数据与实验协议.md)第6、7、9节。


## 1. 从16个振幅到136个实特征

采用 qubit 0 对应基索引最高位的约定。令 $E_a(x)=\bigotimes_j R_Y(a\pi x_j)$，层间纠缠按时间顺序执行 CNOT $0\to1$、$1\to2$、$2\to3$。深度1是 $E_a$，深度2是 $E_a C E_a$，矩阵乘积从右向左作用。

对实振幅态 $\psi\in\mathbb R^{16}$，取特征
\[
\phi(\psi)=(\psi_0^2,\ldots,\psi_{15}^2,\sqrt2\psi_0\psi_1,\ldots,\sqrt2\psi_{14}\psi_{15}).
\]
维数是 $16+16\times15/2=136$，且
\[
\phi(\psi)^\top\phi(\varphi)
=\operatorname{Tr}(\psi\psi^\top\varphi\varphi^\top)
=|\langle\psi|\varphi\rangle|^2.
\]
这说明当前小线路存在可直接计算的经典表示。一般复振幅4比特态可用256维实Hermitian坐标，本单元只检验实振幅情形。下列一致性不能证明量子计算优势。


In [1]:
import os
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[variable] = "4"

from pathlib import Path
import sys
import json
import numpy as np

# 只定位课程内的公开数值模块；不读取行情、账户或外部网络。
for base in (Path.cwd(), *Path.cwd().parents):
    module_dir = base / "experiments" / "feasibility"
    if (module_dir / "quantum_core.py").is_file():
        if str(module_dir) not in sys.path:
            sys.path.insert(0, str(module_dir))
        break
else:
    raise FileNotFoundError("请从课程根目录或 notebooks 子目录运行本单元")

from quantum_core import state_features, fidelity_kernel, real_density_features

# 解析构造的数学输入，不是历史行情；8条训练输入、3条评价输入。
X = 0.15 * np.sin(np.arange(44, dtype=float).reshape(11, 4) * 0.37 + 0.2)

report = []
for name, a, depth in (("Q1", 0.5, 1), ("Q2", 1.0, 1),
                        ("Q3", 0.5, 2), ("Q4", 1.0, 2)):
    states = state_features(X, a=a, depth=depth)
    K = fidelity_kernel(states)
    features = real_density_features(states)
    error = float(np.max(np.abs(K - features @ features.T)))
    norm_error = float(np.max(np.abs(np.sum(np.abs(states)**2, axis=1) - 1.0)))
    assert states.shape == (11, 16)
    assert features.shape == (11, 136)
    assert error < 1e-10 and norm_error < 1e-10
    item = {"candidate": name, "state_shape": list(states.shape),
            "feature_shape": list(features.shape), "kernel_max_error": error,
            "normalization_max_error": norm_error}
    if depth == 1:
        analytic = np.prod(
            np.cos(a * np.pi * (X[:, None, :] - X[None, :, :]) / 2.0)**2, axis=2)
        item["product_formula_max_error"] = float(np.max(np.abs(K - analytic)))
        assert item["product_formula_max_error"] < 1e-10
    report.append(item)
print(json.dumps({"scope": "analytic_math_only", "numpy_version": np.__version__,
                  "candidates": report}, ensure_ascii=False, indent=2))


{
  "scope": "analytic_math_only",
  "numpy_version": "2.3.5",
  "candidates": [
    {
      "candidate": "Q1",
      "state_shape": [
        11,
        16
      ],
      "feature_shape": [
        11,
        136
      ],
      "kernel_max_error": 9.992007221626409e-16,
      "normalization_max_error": 4.440892098500626e-16,
      "product_formula_max_error": 1.1102230246251565e-15
    },
    {
      "candidate": "Q2",
      "state_shape": [
        11,
        16
      ],
      "feature_shape": [
        11,
        136
      ],
      "kernel_max_error": 9.992007221626409e-16,
      "normalization_max_error": 4.440892098500626e-16,
      "product_formula_max_error": 1.3322676295501878e-15
    },
    {
      "candidate": "Q3",
      "state_shape": [
        11,
        16
      ],
      "feature_shape": [
        11,
        136
      ],
      "kernel_max_error": 7.771561172376096e-16,
      "normalization_max_error": 6.661338147750939e-16
    },
    {
      "candidate": "Q4",
     

## 2. 一次有限采样可以破坏半正定性

本例只用 Q3：$a=0.5,L=2$，训练8条、评价3条，每个元素32 shots。训练非对角上三角有28项，交叉核有24项，合计52个测量动作、**1,664个名义shots**。理想单位对角线直接设为1；同一元素镜像填入，不重复收费。

这里直接抽二项计数，无需逐shot运行线路。种子预先固定为20261109。这一解析输入与种子被用于演示“不定性可以发生”，**不是对其发生概率的无偏调查**。独立候选或动作没有共享物理读数；本单元也没有候选筛选或预算优化。

上下三角对称、元素位于 $[0,1]$ 和单位对角线，都不足以保证整个矩阵半正定。


In [2]:
import os
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[variable] = "4"

from pathlib import Path
import sys
import json
import numpy as np

# 只定位课程内的公开数值模块；不读取行情、账户或外部网络。
for base in (Path.cwd(), *Path.cwd().parents):
    module_dir = base / "experiments" / "feasibility"
    if (module_dir / "quantum_core.py").is_file():
        if str(module_dir) not in sys.path:
            sys.path.insert(0, str(module_dir))
        break
else:
    raise FileNotFoundError("请从课程根目录或 notebooks 子目录运行本单元")

from quantum_core import state_features, fidelity_kernel, real_density_features

# 解析构造的数学输入，不是历史行情；8条训练输入、3条评价输入。
X = 0.15 * np.sin(np.arange(44, dtype=float).reshape(11, 4) * 0.37 + 0.2)
n, v, shots = 8, 3, 32
train_states = state_features(X[:n], a=0.5, depth=2)
eval_states = state_features(X[n:], a=0.5, depth=2)
K_exact = fidelity_kernel(train_states)
B_exact = fidelity_kernel(eval_states, train_states)

# 精确概率只用于本地理想采样器；以下修正/拟合只消费采样矩阵。
rng = np.random.default_rng(20261109)
upper = np.triu_indices(n, k=1)
training_successes = rng.binomial(shots, np.clip(K_exact[upper], 0.0, 1.0))
cross_successes = rng.binomial(shots, np.clip(B_exact, 0.0, 1.0))
K_sampled = np.eye(n)
K_sampled[upper] = training_successes / shots
K_sampled[(upper[1], upper[0])] = training_successes / shots
B_sampled = cross_successes / shots

exact_eigenvalues = np.linalg.eigvalsh(K_exact)
sampled_eigenvalues = np.linalg.eigvalsh(K_sampled)
actions = n * (n - 1) // 2 + n * v
assert actions == 52
assert exact_eigenvalues.min() >= -1e-10
assert sampled_eigenvalues.min() < -1e-8
assert np.array_equal(K_sampled, K_sampled.T)
print(json.dumps({
    "scope": "analytic_input_ideal_binomial_sampling",
    "shots_per_element": shots, "measurement_actions": actions,
    "nominal_shots": actions * shots, "actual_cloud_shots": 0,
    "exact_eigenvalues": exact_eigenvalues.tolist(),
    "sampled_eigenvalues": sampled_eigenvalues.tolist(),
    "training_success_counts_upper_triangle": training_successes.tolist(),
    "cross_success_counts": cross_successes.tolist()
}, ensure_ascii=False, indent=2))


{
  "scope": "analytic_input_ideal_binomial_sampling",
  "shots_per_element": 32,
  "measurement_actions": 52,
  "nominal_shots": 1664,
  "actual_cloud_shots": 0,
  "exact_eigenvalues": [
    1.3921266849579767e-06,
    4.95913584629232e-05,
    7.454087929921044e-05,
    0.0031775033478259264,
    0.015783370322322538,
    0.15171367962996068,
    0.8665428710547836,
    6.962657051280656
  ],
  "sampled_eigenvalues": [
    -0.15357964378809474,
    -0.08058405204150805,
    -0.018581968825203514,
    0.044239102321669825,
    0.09660133753373025,
    0.24011635328535325,
    1.0513879211642583,
    6.820400950349793
  ],
  "training_success_counts_upper_triangle": [
    31,
    22,
    17,
    32,
    30,
    24,
    19,
    25,
    25,
    30,
    32,
    28,
    20,
    30,
    23,
    25,
    32,
    30,
    21,
    22,
    31,
    32,
    31,
    27,
    27,
    26,
    21,
    31
  ],
  "cross_success_counts": [
    [
      30,
      28,
      30,
      29,
      31,
      30,
 

## 3. 相容修正，再进行平均损失KRR

设采样训练核为 $\widehat K$，评价交叉核为 $\widehat B$。保留大于阈值 $\tau=10^{-8}\max(1,\max|d|)$ 的训练特征值：
\[
Z_{\mathrm{tr}}=Q_+D_+^{1/2},\qquad
Z_{\mathrm{va}}=\widehat BQ_+D_+^{-1/2}.
\]
随后两者都减去训练特征均值，并除以训练中心化特征平均平方范数的平方根。这样得到相容的训练核与交叉核；不能只裁剪训练核而保留未变的交叉核。

标签 $y_i=\sin(0.4i)$ 也只是数学构造。使用平均平方损失约定和固定 $\lambda=0.01$：
\[
\alpha=(Z_{\mathrm{tr}}Z_{\mathrm{tr}}^\top+n\lambda I)^{-1}(y-\bar y),
\qquad
\widehat y=Z_{\mathrm{va}}Z_{\mathrm{tr}}^\top\alpha+\bar y.
\]
显式特征ridge应给出相同预测。以下用NumPy展开[数值模块](../experiments/feasibility/kernel_models.py)的相同公式；若环境已有SciPy，会额外与该模块比较，否则如实标记未执行该项模块交叉检查。谱阈值边界不保证可微；修正后的核也不再具有原始二项估计的逐元素无偏性。


In [3]:
import os
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[variable] = "4"

from pathlib import Path
import sys
import json
import numpy as np

# 只定位课程内的公开数值模块；不读取行情、账户或外部网络。
for base in (Path.cwd(), *Path.cwd().parents):
    module_dir = base / "experiments" / "feasibility"
    if (module_dir / "quantum_core.py").is_file():
        if str(module_dir) not in sys.path:
            sys.path.insert(0, str(module_dir))
        break
else:
    raise FileNotFoundError("请从课程根目录或 notebooks 子目录运行本单元")

from quantum_core import state_features, fidelity_kernel, real_density_features

# 解析构造的数学输入，不是历史行情；8条训练输入、3条评价输入。
X = 0.15 * np.sin(np.arange(44, dtype=float).reshape(11, 4) * 0.37 + 0.2)
n, v, shots = 8, 3, 32
train_states = state_features(X[:n], a=0.5, depth=2)
eval_states = state_features(X[n:], a=0.5, depth=2)
K_exact = fidelity_kernel(train_states)
B_exact = fidelity_kernel(eval_states, train_states)

# 精确概率只用于本地理想采样器；以下修正/拟合只消费采样矩阵。
rng = np.random.default_rng(20261109)
upper = np.triu_indices(n, k=1)
training_successes = rng.binomial(shots, np.clip(K_exact[upper], 0.0, 1.0))
cross_successes = rng.binomial(shots, np.clip(B_exact, 0.0, 1.0))
K_sampled = np.eye(n)
K_sampled[upper] = training_successes / shots
K_sampled[(upper[1], upper[0])] = training_successes / shots
B_sampled = cross_successes / shots

# 独立新进程也可运行：采样输入已在本单元重新定义。
d, Q = np.linalg.eigh((K_sampled + K_sampled.T) / 2.0)
threshold = 1e-8 * max(1.0, float(np.max(np.abs(d))))
keep = d > threshold
assert keep.any()
Qp, dp = Q[:, keep], d[keep]
Z_train = Qp * np.sqrt(dp)
Z_eval = (B_sampled @ Qp) / np.sqrt(dp)
training_mean = Z_train.mean(axis=0)
Z_train, Z_eval = Z_train - training_mean, Z_eval - training_mean
scale = float(np.mean(np.sum(Z_train * Z_train, axis=1)))
assert scale > 1e-12
Z_train, Z_eval = Z_train / np.sqrt(scale), Z_eval / np.sqrt(scale)
K_repaired, B_repaired = Z_train @ Z_train.T, Z_eval @ Z_train.T

# 另从投影交叉核出发检查中心化与训练尺度。
K_positive = (Qp * dp) @ Qp.T
B_projected = B_sampled @ Qp @ Qp.T
B_expected = (B_projected - B_projected.mean(axis=1)[:, None]
              - K_positive.mean(axis=0)[None, :] + K_positive.mean()) / scale
cross_error = float(np.max(np.abs(B_repaired - B_expected)))
assert cross_error < 1e-10

y = np.sin(0.4 * np.arange(n))
regularization = 0.01
alpha = np.linalg.solve(K_repaired + n * regularization * np.eye(n), y - y.mean())
prediction = B_repaired @ alpha + y.mean()
beta = np.linalg.solve(Z_train.T @ Z_train
                       + n * regularization * np.eye(Z_train.shape[1]),
                       Z_train.T @ (y - y.mean()))
primal_prediction = Z_eval @ beta + y.mean()
duality_error = float(np.max(np.abs(prediction - primal_prediction)))
assert duality_error < 1e-10
joint_features = np.vstack((Z_train, Z_eval))
joint_minimum = float(np.linalg.eigvalsh(joint_features @ joint_features.T).min())
assert joint_minimum >= -1e-10

import importlib.util
module_check = {"status": "not_run", "reason": "SciPy unavailable; NumPy formula checks completed"}
if importlib.util.find_spec("scipy") is not None:
    from kernel_models import kernel_map, predict_mapped
    km_train, km_eval, _ = kernel_map(K_sampled, B_sampled)
    km_prediction = predict_mapped(km_train, km_eval, y, regularization)
    module_error = max(float(np.max(np.abs(km_train @ km_train.T - K_repaired))),
                       float(np.max(np.abs(km_eval @ km_train.T - B_repaired))),
                       float(np.max(np.abs(km_prediction - prediction))))
    assert module_error < 1e-10
    module_check = {"status": "passed", "maximum_error": module_error}

print(json.dumps({
    "scope": "analytic_labels_no_financial_interpretation",
    "retained_rank": int(keep.sum()), "training_centered_scale": scale,
    "cross_compatibility_error": cross_error, "primal_dual_prediction_error": duality_error,
    "joint_gram_minimum_eigenvalue": joint_minimum,
    "regularization_mean_loss": regularization, "diagonal_penalty_n_lambda": n * regularization,
    "predictions": prediction.tolist(), "kernel_models_cross_check": module_check
}, ensure_ascii=False, indent=2))


{
  "scope": "analytic_labels_no_financial_interpretation",
  "retained_rank": 5,
  "training_centered_scale": 0.18002674543661612,
  "cross_compatibility_error": 1.1102230246251565e-15,
  "primal_dual_prediction_error": 1.7763568394002505e-15,
  "joint_gram_minimum_eigenvalue": -4.596356698218631e-16,
  "regularization_mean_loss": 0.01,
  "diagonal_penalty_n_lambda": 0.08,
  "predictions": [
    0.7859641580246897,
    0.8636813778774501,
    0.9373031955027832
  ],
  "kernel_models_cross_check": {
    "status": "not_run",
    "reason": "SciPy unavailable; NumPy formula checks completed"
  }
}


## 能据此得出什么

- 已检查本构造下态范数、保真度核、136维特征核与深度1解析式的一致性。
- 已展示一次有限采样产生不定训练核，并检查修正后的训练/评价映射相容及KRR原—对偶预测一致。
- 这些输出不代表金融预测、候选搜索可靠性、器件噪声表现或量子优势。一个固定种子也不能估计失败率。

继续练习：保持输入和修正规则不变，事先指定多个独立种子与shots档，再报告不定核比例、谱修正幅度及预测差异。次数与总预算需完整记录。课程正式实验的参数和数据准入仍以协议为准，不用本例替代真实金融样本。
